In [ ]:
# ConceptOps MVP Demo

**Pipeline:** `video → masks → events → concepts → episode`

This notebook visualizes the output of a completed ConceptOps run:
- Shows basic run metadata
- Summarizes temporal events
- Displays event thumbnails with CLIP labels
- Lets you inspect the frames inside a selected event

In [ ]:
from pathlib import Path
import json

from PIL import Image
import matplotlib.pyplot as plt

# Path to a completed ConceptOps run
RUN_DIR = Path("outputs/hero_sam3")  # change this for other runs

manifest_path = RUN_DIR / "conceptops_manifest.json"
events_path = RUN_DIR / "events.json"
concepts_path = RUN_DIR / "concepts.json"
episode_path = RUN_DIR / "episode.json"

with manifest_path.open() as f:
    manifest = json.load(f)
with events_path.open() as f:
    events_payload = json.load(f)
with concepts_path.open() as f:
    concepts_payload = json.load(f)
with episode_path.open() as f:
    episode = json.load(f)

events = events_payload.get("events", [])
events_concepts = concepts_payload.get("events_concepts", [])

print("Stages:", manifest["stages"])
print("Num frames:", episode["length"]["num_frames"])
print("Num events:", len(events))
print("Num events_concepts:", len(events_concepts))

In [ ]:
# Quick overview of all events
for ev in events:
    print(
        f"event_id={ev['event_id']:02d}, "
        f"frames={ev['start_frame']}-{ev['end_frame']}, "
        f"num_frames={ev['num_frames']}"
    )

In [ ]:
# Event + label summary
concept_by_event = {c["event_id"]: c for c in events_concepts}

for ev in events:
    c = concept_by_event.get(ev["event_id"])
    labels = c["labels"] if c else []
    scores = [round(s, 2) for s in (c["scores"] if c else [])]
    print(
        f"event_id={ev['event_id']:02d}, "
        f"frames={ev['start_frame']}-{ev['end_frame']}, "
        f"labels={labels}, scores={scores}"
    )

In [ ]:
thumbs_dir = RUN_DIR / "thumbnails"

fig_cols = 3
fig_rows = max(1, (len(events_concepts) + fig_cols - 1) // fig_cols)
fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(4 * fig_cols, 4 * fig_rows))
axes = axes.flatten()

for ax in axes[len(events_concepts):]:
    ax.axis("off")

for ax, c in zip(axes, events_concepts):
    thumb_path = Path(c["thumbnail_path"])
    img = Image.open(thumb_path).convert("RGB")
    ax.imshow(img)

    labels = c["labels"]
    scores = [round(s, 2) for s in c["scores"]]
    title = f"Event {c['event_id']}\n" + ", ".join(
        f"{lbl}({sc})" for lbl, sc in zip(labels, scores)
    )
    ax.set_title(title, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("Available event IDs:", [ev["event_id"] for ev in events])

# Choose which event to inspect (after normalization, 0..N-1 will work)
EVENT_ID = events[0]["event_id"]  # or set manually

ev = next((e for e in events if e["event_id"] == EVENT_ID), None)
if ev is None:
    raise ValueError(f"Event {EVENT_ID} not found. Available: {[ev['event_id'] for ev in events]}")

print(ev)

frames_dir = Path(manifest["frames_dir"])
frame_paths = [
    frames_dir / f"frame_{idx+1:06d}.jpg"
    for idx in range(ev["start_frame"], ev["end_frame"] + 1)
]

n = len(frame_paths)
cols = min(6, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes = axes.flatten()

for ax, p in zip(axes, frame_paths):
    img = Image.open(p).convert("RGB")
    ax.imshow(img)
    ax.set_title(p.name, fontsize=8)
    ax.axis("off")

for ax in axes[len(frame_paths):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
masks_dir = Path(manifest["masks_dir"])
mask_paths = [
    masks_dir / f"mask_{idx+1:06d}.jpg"
    for idx in range(ev["start_frame"], ev["end_frame"] + 1)
]

n = len(mask_paths)
cols = min(6, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes = axes.flatten()

for ax, p in zip(axes, mask_paths):
    img = Image.open(p).convert("L")
    ax.imshow(img, cmap="gray")
    ax.set_title(p.name, fontsize=8)
    ax.axis("off")

for ax in axes[len(mask_paths):]:
    ax.axis("off")

plt.tight_layout()
plt.show()